In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [3]:
# data = pd.read_stata(r"datos/ECU_2004m12_BID.dta", convert_categoricals=False) # para bases de stata

data3 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2004/m3/data_orig/per0104.dta", convert_categoricals=False) # para bases de stata
data6 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2004/m6/data_orig/personas0204.dta") # para bases de stata
data9 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2004/m9/data_orig/per0304.dta") # para bases de stata
data12 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2004/m12/data_orig/per12_2004.dta") # para bases de stata

## Revisar los datos

| marzo | junio | septiembre | diciembre |
|-----------|-----------|-----------|-----------|
| rn  | rn  |   |   |
| area  | area  | area  | area  |
| prov  | prov  |   |   |
| ciudad  | ciudad  | ciudad  | ciudad  |
| zona  | zona  | zona  | zona  |
| sector  | sector  | sector  | sector  |
| panelm  | panelm  | mpanel  | panelm  |
| vivienda  | vivienda  | vivienda  | vivienda  |
| hogar  | hogar  | hogar  | hogar  |
| sexo  | sexo  | sexo  | sexo  |
| edad  | edad  | edad  | edad  |
| pe63  | pe63  | pe63  | pe63  |
| fexp  | fexp  | fexp  | fexp  |
| trabajo  | trabajo  | trabajo  | trabajo  |

En esta encuesta tenemos separadas cuatro diferentes bases para cada trimestre, esto cambia la lógica que habíamos tenido hasta ahora así que de aquí en adelante cambiamos algo del código, mantenemos de acuerdo a las etiquetas de las variables pe63 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, estas variables las usamos antes para identificar la condición de trabajo para diferentes meses en encuestas anuales o incompletas donde asumíamos que mantenía el mismo salario si estaba ocupado en ese mes, sin mbargo estas variables tenían el problema de no corresponder de forma exacta con el año o mes de la encuesta. Ahora sin embargo podemos cambiar las suposiciones y solamente asumir que si la variable 'trabajando' que pregunta si el individuo trabajó la semana pasada se cumple vamos a asumir que trabajo durante todo el trimestre, de esta manera podemos mejorar las suposiciones de ocupación mensual, mantenemos la idea de que si el individuo trabajo recibe su ingreso laboral reportado.

In [4]:
columnas = pd.Index(['rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'panelm',
            'vivienda', 'hogar', 'sexo', 'edad', 'pe63', 'trabajo',
            'fexp'])

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [5]:
data3 = data3[columnas]
data6 = data6[columnas]
data9 = data9[columnas.intersection(data9.columns)]
data12 = data12[columnas.intersection(data12.columns)]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado y limpiamos según los valores de ingrl, para mantener ambas variables para cada base consistente

In [6]:
data3['pe63'] = data3['pe63'].apply(lambda x: np.nan if x > 8048 else x)

data6['pe63'] = data6['pe63'].apply(lambda x: np.nan if x > 11000 else x)

data9['pe63'] = data9['pe63'].apply(lambda x: np.nan if x > 7000 else x)

data12['pe63'] = pd.to_numeric(data12['pe63'], errors='coerce')
data12['pe63'] = data12['pe63'].apply(lambda x: np.nan if x > 9350 else x)

In [7]:
data3['ingr'] = data3['pe63']
data6['ingr'] = data6['pe63']
data9['ingr'] = data9['pe63']
data12['ingr'] = data12['pe63']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados la semana pasada, de acuerdo a la variable 'trabajo'

In [9]:
data3['ingr_t1'] = data3.apply(lambda x: x['ingr'] if x['trabajo'] == 1 else np.nan, axis=1)

data6['ingr_t2'] = data6.apply(lambda x: x['ingr'] if x['trabajo'] == 'si' else np.nan, axis=1)

data9['ingr_t3'] = data9.apply(lambda x: x['ingr'] if x['trabajo'] == 'si' else np.nan, axis=1)

data12['ingr_t4'] = data12.apply(lambda x: x['ingr'] if x['trabajo'] == 'si' else np.nan, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [10]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2004]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [11]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [12]:
# Corregimos los códigos para usarlos cómo texto
data3['ciudad'] = data3['ciudad'].apply(str)
data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data3['ciudad_2'] = data3['ciudad'].apply(lambda x: x[:2])

data6['ciudad'] = data6['ciudad'].apply(str)
data6['ciudad'] = data6['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data6['ciudad_2'] = data6['ciudad'].apply(lambda x: x[:2])

data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:2])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:2])

Diccionario ciudades disponibles

In [13]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito'
}

data3['ciudad_asignada'] = data3['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))
data6['ciudad_asignada'] = data6['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))
data9['ciudad_asignada'] = data9['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))
data12['ciudad_asignada'] = data12['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

In [14]:
data12['ciudad_asignada'].value_counts()

ciudad_asignada
Nacional     61706
Guayaquil    10243
Quito         6946
Cuenca        4148
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [15]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [16]:
data3['ipc_t1'] = data3.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data3['ipc_base_t1'] = data3.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data6['ipc_t2'] = data6.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data6['ipc_base_t2'] = data6.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data9['ipc_t3'] = data9.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data9['ipc_base_t3'] = data9.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data12['ipc_t4'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data12['ipc_base_t4'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [17]:
# Calculamos el deflactor
data3['def_t1'] = (data3['ipc_base_t1'] / data3['ipc_t1'])
data6['def_t2'] = (data6['ipc_base_t2'] / data6['ipc_t2'])
data9['def_t3'] = (data9['ipc_base_t3'] / data9['ipc_t3'])
data12['def_t4'] = (data12['ipc_base_t4'] / data12['ipc_t4'])

Ingreso promedio en el trimeste

In [18]:
data3['ingr_t1_r'] = data3['ingr_t1'] * data3['def_t1']
data6['ingr_t2_r'] = data6['ingr_t2'] * data6['def_t2']
data9['ingr_t3_r'] = data9['ingr_t3'] * data9['def_t3']
data12['ingr_t4_r'] = data12['ingr_t4'] * data12['def_t4']

In [19]:
print(data3['ingr_t1_r'].mean())
print(data6['ingr_t2_r'].mean())
print(data9['ingr_t3_r'].mean())
print(data12['ingr_t4_r'].mean())

251.5039980934143
335.1758388630422
302.06939199854395
273.19468270285324


## Regiones

In [20]:
# Corregimos los códigos para usarlos cómo texto
data3['ciudad'] = data3['ciudad'].apply(str)
data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data3['ciudad_2'] = data3['ciudad'].apply(lambda x: x[:2])

data6['ciudad'] = data6['ciudad'].apply(str)
data6['ciudad'] = data6['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data6['ciudad_2'] = data6['ciudad'].apply(lambda x: x[:2])

data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:2])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:2])

In [21]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [22]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
data3['region'] = data3['ciudad_2'].map(codigo_region)

data6['region'] = data6['ciudad_2'].map(codigo_region)

data9['region'] = data9['ciudad_2'].map(codigo_region)

data12['region'] = data12['ciudad_2'].map(codigo_region)

In [23]:
data3['region'].value_counts()

region
Sierra                  31395
Guayas                  10005
Pichincha                6931
Costa, Santo Domingo     6469
Manabí                   6455
Los Ríos                 6151
El Oro                   5801
Amazonía                 4572
Azuay                    4151
Name: count, dtype: int64

In [24]:
data6['region'].value_counts()

region
Guayas                  6877
Pichincha               4764
Sierra                  2860
El Oro                  2394
Amazonía                2134
Azuay                   2027
Manabí                  1588
Los Ríos                 998
Costa, Santo Domingo     662
Name: count, dtype: int64

In [25]:
data9['region'].value_counts()

region
Guayas                  8467
Sierra                  6827
Pichincha               5870
Amazonía                4689
Manabí                  3097
Azuay                   2839
El Oro                  2786
Los Ríos                1805
Costa, Santo Domingo    1139
Name: count, dtype: int64

In [27]:
data12['region'].value_counts()

region
Sierra                  31770
Guayas                  10243
Pichincha                6946
Manabí                   6677
Los Ríos                 6308
Costa, Santo Domingo     6237
El Oro                   6121
Amazonía                 4593
Azuay                    4148
Name: count, dtype: int64

## Calculo ingreso de los hogares

In [28]:
columnas_idef = pd.Index(['rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'vivienda',
       'hogar'])

data3['idef_hogar'] = data3[columnas_idef].astype(str).agg(''.join, axis=1)
data6['idef_hogar'] = data6[columnas_idef].astype(str).agg(''.join, axis=1)
data9['idef_hogar'] = data9[
    columnas_idef.intersection(data9.columns)
    ].astype(str).agg(''.join, axis=1)
data12['idef_hogar'] = data12[
    columnas_idef.intersection(data12.columns)
    ].astype(str).agg(''.join, axis=1)

print(len(data3['idef_hogar'].unique()))
print(len(data6['idef_hogar'].unique()))
print(len(data9['idef_hogar'].unique()))
print(len(data12['idef_hogar'].unique()))

4996
1532
2256
5005


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [29]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [30]:
data3['ingr_t1_h'] = data3.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data6['ingr_t2_h'] = data6.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data9['ingr_t3_h'] = data9.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data12['ingr_t4_h'] = data12.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [31]:
print(data3['ingr_t1_h'].mean())
print(data6['ingr_t2_h'].mean())
print(data9['ingr_t3_h'].mean())
print(data12['ingr_t4_h'].mean())

942.4220002583984
1344.7741236476306
1144.0722630969337
1050.9319997443554


## Sacamos edades negativas y mayores a 100 años

In [32]:
print(len(data3))
print(len(data6))
print(len(data9))
print(len(data12))

81930
24304
37519
83043


Transformamos las variables de edad a numericas para evitar problemas

In [33]:
data3['edad'] = pd.to_numeric(data3['edad'], errors='coerce')
data6['edad'] = pd.to_numeric(data6['edad'], errors='coerce')
data9['edad'] = pd.to_numeric(data9['edad'], errors='coerce')
data12['edad'] = pd.to_numeric(data12['edad'], errors='coerce')

In [34]:
data3 = data3.loc[(data3['edad'] >= 0) & (data3['edad'] < 100)]
data6 = data6.loc[(data6['edad'] >= 0) & (data6['edad'] < 100)]
data9 = data9.loc[(data9['edad'] >= 0) & (data9['edad'] < 100)]
data12 = data12.loc[(data12['edad'] >= 0) & (data12['edad'] < 100)]

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [35]:
k = 0.4
s = 0.9

In [36]:
# Si es necesario calcular el número de niños
data3['es_nino'] = data3['edad'] < 10
data3['ninos'] = data3.groupby('idef_hogar')['es_nino'].transform('sum')

data6['es_nino'] = data6['edad'] < 10
data6['ninos'] = data6.groupby('idef_hogar')['es_nino'].transform('sum')

data9['es_nino'] = data9['edad'] < 10
data9['ninos'] = data9.groupby('idef_hogar')['es_nino'].transform('sum')

data12['es_nino'] = data12['edad'] < 10
data12['ninos'] = data12.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data3['es_adulto'] = data3['edad'] > 10
data3['adultos'] = data3.groupby('idef_hogar')['es_adulto'].transform('sum')

data6['es_adulto'] = data6['edad'] > 10
data6['adultos'] = data6.groupby('idef_hogar')['es_adulto'].transform('sum')

data9['es_adulto'] = data9['edad'] > 10
data9['adultos'] = data9.groupby('idef_hogar')['es_adulto'].transform('sum')

data12['es_adulto'] = data12['edad'] > 10
data12['adultos'] = data12.groupby('idef_hogar')['es_adulto'].transform('sum')

In [37]:
data3['escala'] = (data3['adultos'] + k * data3['ninos']) ** s
data6['escala'] = (data6['adultos'] + k * data6['ninos']) ** s
data9['escala'] = (data9['adultos'] + k * data9['ninos']) ** s
data12['escala'] = (data12['adultos'] + k * data12['ninos']) ** s

In [43]:
data3['ingr_t_t3'] = data3['ingr_t1_h'] / data3['escala']
data6['ingr_t_t6'] = data6['ingr_t2_h'] / data6['escala']
data9['ingr_t_t9'] = data9['ingr_t3_h'] / data9['escala']
data12['ingr_t_t12'] = data12['ingr_t4_h'] / data12['escala']

In [45]:
print(data3['ingr_t_t3'].mean())
print(data6['ingr_t_t6'].mean())
print(data9['ingr_t_t9'].mean())
print(data12['ingr_t_t12'].mean())

84.04828805498796
123.53585065535624
102.74494340606383
93.23843997379294


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [46]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 2004

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [55]:
resultados_list = []

# Para cada trimeste
for t in [3, 6, 9, 12]:
    col_ingr = f'ingr_t_t{t}'
    
    # Selecciona el dataframe correspondiente
    if t == 3:
        df_actual = data3
        umbral = umbral_dict.get(1)
        salario = salario_dict.get(1)
        tr = 1
    elif t == 6:
        df_actual = data6
        umbral = umbral_dict.get(2)
        salario = salario_dict.get(2)
        tr = 2
    elif t == 9:
        df_actual = data9
        umbral = umbral_dict.get(3)
        salario = salario_dict.get(3)
        tr = 3
    elif t == 12:
        df_actual = data12
        umbral = umbral_dict.get(4)
        salario = salario_dict.get(4)
        tr = 4
    else:
        continue

    # Agrupa por región
    grouped = df_actual.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': tr,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana + 1 if mediana < 1 else mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / (mediana + 1 if mediana < 1 else mediana)         
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2004,1,Amazonía,0.570532,0.314135,0.216953,0.093815,0.189067,0.292080,71.334006,53.022095,135.63,2.557990
1,2004,1,Azuay,0.392363,0.167283,0.094797,0.077969,0.153423,0.230235,106.807551,76.868896,135.63,1.764433
2,2004,1,"Costa, Santo Domingo",0.587397,0.286588,0.183458,0.077363,0.155544,0.237381,68.248539,54.104736,135.63,2.506805
3,2004,1,El Oro,0.347752,0.129486,0.071331,0.065315,0.128724,0.195563,106.379819,89.925803,135.63,1.508243
4,2004,1,Guayas,0.455568,0.189223,0.108250,0.085037,0.162900,0.235610,94.769355,66.637028,135.63,2.035355
5,2004,1,Los Ríos,0.542243,0.227898,0.128020,0.051644,0.103452,0.155734,67.962016,57.744905,135.63,2.348779
6,2004,1,Manabí,0.646461,0.346599,0.230114,0.099014,0.192523,0.284398,63.845680,43.155567,135.63,3.142816
7,2004,1,Pichincha,0.201888,0.078634,0.045902,0.075060,0.147054,0.220220,156.246886,121.056685,135.63,1.120384
8,2004,1,Sierra,0.543350,0.275643,0.180598,0.086393,0.174267,0.273820,74.882113,57.440496,135.63,2.361226
9,2004,2,Amazonía,0.230303,0.072760,0.041586,0.052443,0.105970,0.166939,132.350038,112.761052,135.63,1.202809


### Inserta los cálculos en la base final

In [56]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [57]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_final_regional.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_final_regional.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')